# response words check

한글로 응답한 단어들을 영어로 번역하는 과정에서, word2vec에 없는 단어들을 걸러내는 코드입니다. \
need_to_be_fixed.csv 파일에서 확인할 수 있습니다.

In [9]:
import os
import re
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

In [10]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
research1_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'


# data/processed 폴더 위치 지정
processed_data_dir = research1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')
# data/data_preprocess 폴더 위치 지정
data_process_dir = research1_dir + ('\\data\\data_preprocess\\' if os_system == 'Windows' else '/data/data_preprocess/')

In [11]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(research1_dir + model_path, binary=True)

In [12]:
tbl_data = pd.read_csv(processed_data_dir + 'merged_data.csv')
tbl_data

,subject,tear1,tear2,tear3,tear4,tear5,tear6,tear7,tear8,tear9,...,abuse31,abuse32,abuse33,abuse34,abuse35,abuse36,abuse37,abuse38,abuse39,abuse40
0,1,sadness,depressed,coolness,annoyance,failure,upset,conflict,farewell,heartache,...,a strange woman,do not ask assault,crime,detective,drama,kwon ryongi narsha,history drama,king seondeok,female,subjectivity
1,2,sadness,sob,sad ending,movie,helminth,bug,apple,fruit,melon,...,announcement,announcement,recruitment,participant,experiment,lab,research complex,gnme,major,student council
2,3,masterpiece,world,travel,carrier,airport,airplane,sky,cloud,happiness,...,smell,sensitive,myself,singularity,geezer,scientist,difficulty,headache,tylenol,labor pains
3,4,graduated,schoolmaster,black,notebook screen,laver,salted,soy sauce,food,dumpling,...,mirror,face,acne,stress,chronic,pain,hospital,patient wear,blanket,everest
4,5,gloom,sadness,dawn,poem,emotion,emotion,emotion,emotion,emotion,...,lively,life,plan,future,past,history,palace museum,japanese occupation,gyeongbokgung,gwanghwamun
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,346,sadness,blue,aquarium,fish,nemo,cuteness,winnie,doll,poksinham,...,doksagwa,foodpoisoning,summer,sea,beach,drive,car,wheel,bicycle,hanriver
346,347,person,human,emotion,sadness,pleasure,depressed,feeling,exam,midtermexam,...,sonheung-min,ronaldo,goal,messi,argentina,brahow,worldcup,catarrh,arab,dubai
347,348,longing,mistake,hongdae,roomcafe,boardgame,halrigalri,nintendoswitch,animalforest,debt,...,alleyrestaurant,porkcutlet,swings,pizza,pizzaschool,isu,seolah,and3,admissions,beauty
348,349,sadness,depressed,depression,farewell,separation,couple,love,lovekoen,yoshihiroakiyama,...,basketballshoes,jordan,legend,respect,father,healwayssuffer,effort,passion,thewoocheol,friend


In [13]:
seed_words = ['abuse', 'tear', 'mirror', 'family'] #  [눈물, 가족, 거울, 학대]
n_respond_words = 40 # 하나의 시드당 40개의 단어 응답
n_subject = len(tbl_data) #350
except_words = []
for seed_word in seed_words: # abuse, tear, mirror
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # abuse1~40, tear1~40, mirror1~40


    for column in word_columns:
        for i_subject in range(n_subject):
            response_word = tbl_data.iloc[i_subject][column]

            # remove white space after the word such as 'stoma ' and ' stoma'
            tbl_data[column] = tbl_data[column].str.strip()
            
            ##### exceptional cases 해결 
            # Remove special character
            tbl_data[column] = tbl_data[column].apply(lambda x: re.sub('[^\w\s]', '', str(x)) if isinstance(x, (str, np.ndarray)) else x)
            # Lower case로 통일
            tbl_data[column] = tbl_data[column].str.lower()
            

            if isinstance(response_word, str):
                response_word = response_word.split()
                response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]

                for i_el in range( len(response_word) ):
                    try:
                        vec_word2vec_in = word2vec_model[response_word[i_el]]
                    except:
                        except_word = {
                            'subject': tbl_data.iloc[i_subject]['subject'],
                            'category': column,
                            'word': response_word
                        }
                        except_words.append(except_word)


In [14]:
except_words

[{'subject': 141, 'category': 'abuse1', 'word': ['domesticviolence']},
 {'subject': 152, 'category': 'abuse1', 'word': ['domesticviolence']},
 {'subject': 153, 'category': 'abuse1', 'word': ['daycarecenter']},
 {'subject': 156, 'category': 'abuse1', 'word': ['childabuse']},
 {'subject': 162,
  'category': 'abuse1',
  'word': ['disappearfromthisworldhalgeot']},
 {'subject': 169, 'category': 'abuse1', 'word': ['daycarecenter']},
 {'subject': 173, 'category': 'abuse1', 'word': ['childabuse']},
 {'subject': 178, 'category': 'abuse1', 'word': ['violenceagainstchildren']},
 {'subject': 180, 'category': 'abuse1', 'word': ['childabuse']},
 {'subject': 189, 'category': 'abuse1', 'word': ['emotionalabuse']},
 {'subject': 196, 'category': 'abuse1', 'word': ['childabuse']},
 {'subject': 227, 'category': 'abuse1', 'word': ['itsannoying']},
 {'subject': 228, 'category': 'abuse1', 'word': ['socialissues']},
 {'subject': 230, 'category': 'abuse1', 'word': ['domesticviolence']},
 {'subject': 242, 'cate

In [15]:
pd.DataFrame(except_words).to_csv(data_process_dir + 'need_to_be_fixed.csv')